<a href="https://colab.research.google.com/github/lricci03/challengedata/blob/main/CFM2024-new/joint_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Joint model: RNN and engineered features

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Sequential Features and Target

In [36]:
df = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtrain.parquet")

In [37]:
df_num = df[['obs_id','price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'action', 'venue']]
df_num['venue'] = df_num['venue'].apply(str)
df_num = pd.get_dummies(df_num, dtype=float)

/tmp/ipykernel_1301/4138682116.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_num['venue'] = df_num['venue'].apply(str)


In [ ]:
df_num.head()

,obs_id,price,bid,ask,bid_size,ask_size,flux,action_A,action_D,action_U,venue_0,venue_1,venue_2,venue_3,venue_4,venue_5
0,0,0.30,0.0,0.01,100,1,100,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0,-0.17,0.0,0.01,100,1,100,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,0.28,0.0,0.01,100,1,-100,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0,0.30,0.0,0.01,100,1,100,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0,0.37,0.0,0.01,100,1,-100,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Don't need df anymore

In [38]:
import gc

del df
gc.collect()

773

In [7]:
targets = pd.read_csv("/content/drive/MyDrive/Challengedata/y_train_or6m3Ta.csv")

Ensure alignment

In [8]:
targets = targets.set_index("obs_id")
df_num = df_num.set_index("obs_id")

In [ ]:
targets.head()

,eqt_code_cat
obs_id,
0,10
1,15
2,0
3,13
4,0


The first 6 feature positions of df_num are numerical: [0,1,2,3,4,5]

In [ ]:
df_num.iloc[:,0:6]

,price,bid,ask,bid_size,ask_size,flux
obs_id,,,,,,
0,0.30,0.00,0.01,100,1,100
0,-0.17,0.00,0.01,100,1,100
0,0.28,0.00,0.01,100,1,-100
0,0.30,0.00,0.01,100,1,100
0,0.37,0.00,0.01,100,1,-100
...,...,...,...,...,...,...
160799,1.32,0.01,0.06,735,261,-100
160799,0.06,0.01,0.06,735,361,100
160799,1.26,0.01,0.06,735,361,100


Reshape X. This reinterprets every consecutive block of 100 rows as one sample. The -1 means: infer this dimension automatically.

In [39]:
import torch
import torch.nn as nn

X_seq = torch.tensor(df_num.values, dtype=torch.float32)
X_seq = X_seq.reshape(-1, 100, 15)
y = torch.tensor(targets.values.squeeze(), dtype=torch.long)

NameError: name 'targets' is not defined

Delete the dataframes

In [ ]:
del df_num, targets
gc.collect()

## Engineered Features

In [11]:
df_eng = pd.read_csv("/content/drive/MyDrive/Challengedata/eng_features.csv")

In [ ]:
df_eng.head()

,obs_id,median_price,median_mid_price,mean_spread,median_spread,std_spread,min_spread,max_spread,mean_bid_size,median_bid_size,...,std_flux,add_ratio,delete_ratio,update_ratio,venue_4_ratio,venue_1_ratio,venue_5_ratio,venue_2_ratio,venue_0_ratio,venue_3_ratio
0,0,0.00,0.010,0.0152,0.02,0.005021,0.01,0.02,190.60,100.0,...,93.739861,0.51,0.49,0.00,0.59,0.14,0.07,0.13,0.05,0.02
1,1,0.01,0.040,0.0392,0.04,0.006917,0.02,0.06,290.94,300.0,...,94.011927,0.49,0.43,0.08,0.51,0.07,0.21,0.10,0.06,0.05
2,2,0.01,0.005,0.0107,0.01,0.002564,0.01,0.02,1902.15,1510.0,...,151.864373,0.54,0.46,0.00,0.44,0.02,0.19,0.01,0.11,0.23
3,3,0.21,0.150,0.1310,0.10,0.046482,0.10,0.20,275.13,200.0,...,74.986447,0.50,0.49,0.01,0.43,0.18,0.09,0.11,0.18,0.01
4,4,0.02,0.020,0.0174,0.02,0.004408,0.01,0.02,143.15,122.5,...,126.849955,0.48,0.52,0.00,0.42,0.09,0.18,0.02,0.19,0.10


In [12]:
df_eng = df_eng.set_index("obs_id")

In [13]:
X_eng = torch.tensor(df_eng.values, dtype=torch.float32)

In [14]:
X_seq.shape, X_eng.shape, y.shape

(torch.Size([160800, 100, 15]), torch.Size([160800, 30]), torch.Size([160800]))

## Train and Validation set

In [40]:
X_seq_train = X_seq[:128640]
X_seq_valid = X_seq[128640:]

X_eng_train = X_eng[:128640]
X_eng_valid = X_eng[128640:]

y_train = y[:128640]
y_valid = y[128640:]

### Scaling engineered features

We scale the engineered features.

In [16]:
torch.isnan(X_eng).sum()

tensor(0)

Compute mean and standard deviation of the test set, then apply the scaling to both train and validation.

In [17]:
mean = X_eng_train.mean(dim=0, keepdim=True)
std = X_eng_train.std(dim=0, keepdim=True)

# avoid division by zero for constant features
std[std == 0] = 1.0

X_eng_train = (X_eng_train - mean) / std
X_eng_valid = (X_eng_valid - mean) / std

### Scaling sequential features

We scale **some of the** sequential features, only the numerical ones (not one-hot-encoded categorical ones).

The indices of the numerical features are [0,1,2,3,4,5]. [0,1,2] are price, bid, ask.
We scale **only bid_size, ask_size, flux**: [3,4,5]

In [29]:
num_idx = [3, 4, 5]

In [30]:
mean = X_seq_train[:, :, num_idx].mean(dim=(0, 1), keepdim=True)
std = X_seq_train[:, :, num_idx].std(dim=(0, 1), keepdim=True)

std[std == 0] = 1.0

In [31]:
X_seq_train[:, :, num_idx] = (X_seq_train[:, :, num_idx] - mean) / std

X_seq_valid[:, :, num_idx] = (X_seq_valid[:, :, num_idx] - mean) / std

## Dataset and DataLoader

In [18]:
from torch.utils.data import Dataset
import torch

class StockDataset(Dataset):
    def __init__(self, X_seq, X_eng, y):
        self.X_seq = X_seq
        self.X_eng = X_eng
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_eng[idx],
            self.y[idx]
        )

In [41]:
train_dataset = StockDataset(X_seq_train, X_eng_train, y_train)
valid_dataset = StockDataset(X_seq_valid, X_eng_valid, y_valid)

In [42]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
valid_loader = DataLoader(valid_dataset, batch_size=64)

## Joint GRU model

In [24]:
import torch
import torch.nn as nn

class JointGRUModel(nn.Module):
    def __init__(self, n_seq_features, n_engineered_features, hidden_size=128, n_classes=24):
        super().__init__()

        self.gru = nn.GRU(
            input_size=n_seq_features,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.classifier = nn.Linear(
            hidden_size + n_engineered_features,
            n_classes
        )

    def forward(self, x_seq, x_eng):

        # x_seq: (batch_size, 100, n_seq_features)

        outputs, last_state = self.gru(x_seq)

        # For a 1-layer GRU:
        # outputs shape: [batch_size, window length (100), hidden_size)
        # outputs[:, -1] shape: [batch_size, hidden_size]
        h_gru = outputs[:, -1]

        # x_eng:
        # (batch_size, n_engineered_features)

        x = torch.cat([h_gru, x_eng], dim=1)

        # x:
        # (batch_size, 128 + n_engineered_features)

        logits = self.classifier(x)

        return logits

## Train and evaluation functions

We need to modify slightly since the batches are now of the form: x_seq, x_eng, y instead of previously: x y

In [21]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.1 MB/s eta 0:00:00


In [23]:
import torchmetrics
import copy

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_seq, X_eng, y in data_loader:
            X_seq, X_eng, y = X_seq.to(device), X_eng.to(device), y.to(device)

            logits = model(X_seq, X_eng)

            y_class = logits.argmax(dim=1)
            metric.update(y_class, y)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):

    best_valid = -float("inf")
    best_model_state = None
    best_epoch = None

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max", # we monitor accuracy
        patience=patience,
        factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_seq, X_eng, y in train_loader:
            X_seq, X_eng, y = X_seq.to(device), X_eng.to(device), y.to(device)

            logits = model(X_seq, X_eng)

            loss = loss_fn(logits, y)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = logits.argmax(dim=1)
            metric.update(y_class, y)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        val_metric = evaluate_tm(model, valid_loader, metric).item()

        if val_metric > best_valid:
          best_valid = val_metric
          best_model_state = copy.deepcopy(model.state_dict())
          best_epoch = epoch +1 # +1 bc epoch starts at 0

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(val_metric)

        scheduler.step(val_metric)

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")

    model.load_state_dict(best_model_state)
    print("Best epoch:", best_epoch)
    print("Best validation accuracy:", best_valid)

    return history

## Train the model

In [27]:
torch.manual_seed(42)

joint_model = JointGRUModel(n_seq_features=15, n_engineered_features=30, hidden_size=128, n_classes=24).to(device)

check the gpu before training

In [24]:
torch.cuda.is_available()

True

In [29]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(joint_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(joint_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.6676, train metric: 0.1602, valid metric: 0.2369
Epoch 2/50, train loss: 2.3125, train metric: 0.2558, valid metric: 0.2740
Epoch 3/50, train loss: 2.2220, train metric: 0.2794, valid metric: 0.2863
Epoch 4/50, train loss: 2.1716, train metric: 0.2920, valid metric: 0.2973
Epoch 5/50, train loss: 2.1387, train metric: 0.2993, valid metric: 0.3016
Epoch 6/50, train loss: 2.1147, train metric: 0.3053, valid metric: 0.3093
Epoch 7/50, train loss: 2.0961, train metric: 0.3108, valid metric: 0.3136
Epoch 8/50, train loss: 2.0805, train metric: 0.3146, valid metric: 0.3159
Epoch 9/50, train loss: 2.0658, train metric: 0.3203, valid metric: 0.3157
Epoch 10/50, train loss: 2.0516, train metric: 0.3247, valid metric: 0.3278
Epoch 11/50, train loss: 2.0361, train metric: 0.3293, valid metric: 0.3329
Epoch 12/50, train loss: 2.0192, train metric: 0.3359, valid metric: 0.3345
Epoch 13/50, train loss: 1.9984, train metric: 0.3420, valid metric: 0.3435
Epoch 14/50, train lo

In [30]:
history2 = train(joint_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=20)

Epoch 1/20, train loss: 1.6090, train metric: 0.4550, valid metric: 0.4429
Epoch 2/20, train loss: 1.6022, train metric: 0.4567, valid metric: 0.4486
Epoch 3/20, train loss: 1.5961, train metric: 0.4588, valid metric: 0.4508
Epoch 4/20, train loss: 1.5893, train metric: 0.4603, valid metric: 0.4512
Epoch 5/20, train loss: 1.5837, train metric: 0.4624, valid metric: 0.4540
Epoch 6/20, train loss: 1.5776, train metric: 0.4641, valid metric: 0.4503
Epoch 7/20, train loss: 1.5713, train metric: 0.4667, valid metric: 0.4506
Epoch 8/20, train loss: 1.5657, train metric: 0.4676, valid metric: 0.4564
Epoch 9/20, train loss: 1.5599, train metric: 0.4702, valid metric: 0.4570
Epoch 10/20, train loss: 1.5533, train metric: 0.4705, valid metric: 0.4562
Epoch 11/20, train loss: 1.5483, train metric: 0.4738, valid metric: 0.4613
Epoch 12/20, train loss: 1.5423, train metric: 0.4751, valid metric: 0.4628
Epoch 13/20, train loss: 1.5367, train metric: 0.4758, valid metric: 0.4617
Epoch 14/20, train lo

## Engineered Only Model

In [ ]:
class EngOnlyModel(nn.Module):
    def __init__(self, n_engineered_features, n_classes=24):
        super().__init__()
        self.classifier = nn.Linear(n_engineered_features, n_classes)

    def forward(self, x_eng):
        return self.classifier(x_eng)

In [ ]:
import torchmetrics

def evaluate_tm2(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)

            y_class = y_pred.argmax(dim=1)
            metric.update(y_class, y_batch)
    return metric.compute()

def train2(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max", # we monitor accuracy
        patience=patience,
        factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        val_metric = evaluate_tm2(model, valid_loader, metric).item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(val_metric)

        scheduler.step(val_metric)

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
print(X_eng_train.shape, y_train.shape)
print(X_eng_valid.shape, y_valid.shape)

torch.Size([128640, 30]) torch.Size([128640])
torch.Size([32160, 30]) torch.Size([32160])


In [ ]:
from torch.utils.data import TensorDataset

train_eng_only_dataset = TensorDataset(X_eng_train, y_train)
valid_eng_only_dataset = TensorDataset(X_eng_valid, y_valid)

In [ ]:
from torch.utils.data import DataLoader

train_eng_only_loader = DataLoader(train_eng_only_dataset, batch_size=64, shuffle=True)
valid_eng_only_loader = DataLoader(valid_eng_only_dataset, batch_size=32)

In [ ]:
torch.manual_seed(42)

eng_only_model = EngOnlyModel(n_engineered_features=30, n_classes=24).to(device)

In [ ]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(eng_only_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train2(eng_only_model, optimizer, criterion, accuracy, train_eng_only_loader,
                valid_eng_only_loader, n_epochs=50)

Epoch 1/50, train loss: 3.0480, train metric: 0.1025, valid metric: 0.1532
Epoch 2/50, train loss: 2.7929, train metric: 0.1765, valid metric: 0.1935
Epoch 3/50, train loss: 2.6746, train metric: 0.2034, valid metric: 0.2080
Epoch 4/50, train loss: 2.6076, train metric: 0.2144, valid metric: 0.2178
Epoch 5/50, train loss: 2.5654, train metric: 0.2220, valid metric: 0.2238
Epoch 6/50, train loss: 2.5363, train metric: 0.2276, valid metric: 0.2277
Epoch 7/50, train loss: 2.5147, train metric: 0.2312, valid metric: 0.2318
Epoch 8/50, train loss: 2.4980, train metric: 0.2343, valid metric: 0.2350
Epoch 9/50, train loss: 2.4845, train metric: 0.2368, valid metric: 0.2373
Epoch 10/50, train loss: 2.4735, train metric: 0.2393, valid metric: 0.2393
Epoch 11/50, train loss: 2.4641, train metric: 0.2416, valid metric: 0.2407
Epoch 12/50, train loss: 2.4562, train metric: 0.2426, valid metric: 0.2428
Epoch 13/50, train loss: 2.4492, train metric: 0.2442, valid metric: 0.2439
Epoch 14/50, train lo

## Joint 2-GRU model

In [25]:
import torch
import torch.nn as nn

class Joint2GRUModel(nn.Module):
    def __init__(self, n_seq_features, n_engineered_features, hidden_size=128, n_classes=24):
        super().__init__()

        self.gru = nn.GRU(input_size=n_seq_features, hidden_size=hidden_size, num_layers=2, batch_first=True)

        self.classifier = nn.Linear(
            hidden_size + n_engineered_features,
            n_classes
        )

    def forward(self, x_seq, x_eng):

        # x_seq: (batch_size, 100, n_seq_features)

        outputs, last_state = self.gru(x_seq)

        # For a 1-layer GRU:
        # outputs shape: [batch_size, window length (100), hidden_size)
        # outputs[:, -1] shape: [batch_size, hidden_size]
        h_gru = outputs[:, -1]

        # x_eng:
        # (batch_size, n_engineered_features)

        x = torch.cat([h_gru, x_eng], dim=1)

        # x:
        # (batch_size, 128 + n_engineered_features)

        logits = self.classifier(x)

        return logits

In [26]:
torch.manual_seed(42)

joint_model_2 = Joint2GRUModel(n_seq_features=15, n_engineered_features=30, hidden_size=128, n_classes=24).to(device)

torch.manual_seed(42)

optimizer = torch.optim.NAdam(joint_model_2.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(joint_model_2, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.6972, train metric: 0.1527, valid metric: 0.2194
Epoch 2/50, train loss: 2.4308, train metric: 0.2341, valid metric: 0.2497
Epoch 3/50, train loss: 2.3355, train metric: 0.2562, valid metric: 0.2641
Epoch 4/50, train loss: 2.2744, train metric: 0.2701, valid metric: 0.2796
Epoch 5/50, train loss: 2.2225, train metric: 0.2837, valid metric: 0.2875
Epoch 6/50, train loss: 2.1788, train metric: 0.2950, valid metric: 0.2964
Epoch 7/50, train loss: 2.1439, train metric: 0.3056, valid metric: 0.2989
Epoch 8/50, train loss: 2.1142, train metric: 0.3118, valid metric: 0.3105
Epoch 9/50, train loss: 2.0869, train metric: 0.3204, valid metric: 0.3175
Epoch 10/50, train loss: 2.0619, train metric: 0.3276, valid metric: 0.3268
Epoch 11/50, train loss: 2.0333, train metric: 0.3375, valid metric: 0.3285
Epoch 12/50, train loss: 2.0037, train metric: 0.3481, valid metric: 0.3332
Epoch 13/50, train loss: 1.9739, train metric: 0.3562, valid metric: 0.3532
Epoch 14/50, train lo

In [27]:
history_2 = train(joint_model_2, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 1.3633, train metric: 0.5424, valid metric: 0.5125
Epoch 2/50, train loss: 1.3530, train metric: 0.5441, valid metric: 0.5065
Epoch 3/50, train loss: 1.3426, train metric: 0.5476, valid metric: 0.5110
Epoch 4/50, train loss: 1.3316, train metric: 0.5511, valid metric: 0.5229
Epoch 5/50, train loss: 1.3206, train metric: 0.5542, valid metric: 0.5199
Epoch 6/50, train loss: 1.3101, train metric: 0.5579, valid metric: 0.5274
Epoch 7/50, train loss: 1.3008, train metric: 0.5604, valid metric: 0.5273
Epoch 8/50, train loss: 1.2900, train metric: 0.5636, valid metric: 0.5294
Epoch 9/50, train loss: 1.2813, train metric: 0.5666, valid metric: 0.5288
Epoch 10/50, train loss: 1.2707, train metric: 0.5692, valid metric: 0.5275
Epoch 11/50, train loss: 1.2621, train metric: 0.5738, valid metric: 0.5305
Epoch 12/50, train loss: 1.2528, train metric: 0.5761, valid metric: 0.5362
Epoch 13/50, train loss: 1.2445, train metric: 0.5805, valid metric: 0.5308
Epoch 14/50, train lo

### Save and load the model

In [28]:
torch.save(
    joint_model_2.state_dict(),
    "/content/drive/MyDrive/Challengedata/deep_joint_model.pt"
)

In [ ]:
joint_model_2.load_state_dict(
    torch.load("/content/drive/MyDrive/Challengedata/deep_joint_model.pt")
)

## Joint 2-GRU trained on normalized data

We run the normalization cells for the sequential inputs, and re-run the dataset and dataloader.

In [34]:
torch.manual_seed(42)

joint_model_3 = Joint2GRUModel(n_seq_features=15, n_engineered_features=30, hidden_size=128, n_classes=24).to(device)

torch.manual_seed(42)

optimizer = torch.optim.NAdam(joint_model_3.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(joint_model_3, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 2.5466, train metric: 0.1738, valid metric: 0.2441
Epoch 2/50, train loss: 2.2719, train metric: 0.2637, valid metric: 0.2802
Epoch 3/50, train loss: 2.1873, train metric: 0.2901, valid metric: 0.2968
Epoch 4/50, train loss: 2.1340, train metric: 0.3035, valid metric: 0.3089
Epoch 5/50, train loss: 2.0952, train metric: 0.3131, valid metric: 0.3121
Epoch 6/50, train loss: 2.0577, train metric: 0.3245, valid metric: 0.3265
Epoch 7/50, train loss: 2.0150, train metric: 0.3368, valid metric: 0.3389
Epoch 8/50, train loss: 1.9837, train metric: 0.3438, valid metric: 0.3372
Epoch 9/50, train loss: 1.9605, train metric: 0.3490, valid metric: 0.3447
Epoch 10/50, train loss: 1.9421, train metric: 0.3546, valid metric: 0.3522
Epoch 11/50, train loss: 1.9248, train metric: 0.3596, valid metric: 0.3589
Epoch 12/50, train loss: 1.9081, train metric: 0.3649, valid metric: 0.3618
Epoch 13/50, train loss: 1.8914, train metric: 0.3701, valid metric: 0.3687
Epoch 14/50, train lo

The validation accuracy is comparable as training on non-normalized sequential data.

The joint model performs worse than the sequential-only model.

## Deep Joint 2-GRU

Construct a deep joint model, where by 'deep' we mean that we add a top linear layer when combining sequential and engineered data.

In [43]:
import torch
import torch.nn as nn

class DeepJoint2GRUModel(nn.Module):
    def __init__(self, n_seq_features, n_engineered_features, hidden_size=128, n_classes=24):
        super().__init__()

        self.gru = nn.GRU(input_size=n_seq_features, hidden_size=hidden_size, num_layers=2, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + n_engineered_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
            )

    def forward(self, x_seq, x_eng):

        # x_seq: (batch_size, 100, n_seq_features)

        outputs, last_state = self.gru(x_seq)

        # For a 1-layer GRU:
        # outputs shape: [batch_size, window length (100), hidden_size)
        # outputs[:, -1] shape: [batch_size, hidden_size]
        h_gru = outputs[:, -1]

        # x_eng:
        # (batch_size, n_engineered_features)

        x = torch.cat([h_gru, x_eng], dim=1)

        # x:
        # (batch_size, 128 + n_engineered_features)

        logits = self.classifier(x)

        return logits

Deep Joint 2Gru model trained with non-normalized sequential inputs.

In [45]:
torch.manual_seed(42)

deep_joint_model = DeepJoint2GRUModel(n_seq_features=15, n_engineered_features=30, hidden_size=128, n_classes=24).to(device)

torch.manual_seed(42)

optimizer = torch.optim.NAdam(deep_joint_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(deep_joint_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.0123, train metric: 0.1110, valid metric: 0.1436
Epoch 2/50, train loss: 2.7101, train metric: 0.1440, valid metric: 0.1536
Epoch 3/50, train loss: 2.6501, train metric: 0.1585, valid metric: 0.1701
Epoch 4/50, train loss: 2.6188, train metric: 0.1675, valid metric: 0.1368
Epoch 5/50, train loss: 2.5975, train metric: 0.1736, valid metric: 0.1714
Epoch 6/50, train loss: 2.5699, train metric: 0.1781, valid metric: 0.1782
Epoch 7/50, train loss: 2.5518, train metric: 0.1846, valid metric: 0.1863
Epoch 8/50, train loss: 2.5378, train metric: 0.1882, valid metric: 0.1868
Epoch 9/50, train loss: 2.5269, train metric: 0.1909, valid metric: 0.1933
Epoch 10/50, train loss: 2.5134, train metric: 0.1932, valid metric: 0.1999
Epoch 11/50, train loss: 2.5046, train metric: 0.1969, valid metric: 0.2012
Epoch 12/50, train loss: 2.4954, train metric: 0.2009, valid metric: 0.1951
Epoch 13/50, train loss: 2.4861, train metric: 0.2035, valid metric: 0.2001
Epoch 14/50, train lo